In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as st

In [2]:
def cohen_glass_hedges(x,y):
    nx = len(x)
    ny = len(y)
    pstdnmr = (nx-1)*x.var(ddof=1) + (ny-1)*y.var(ddof=1)   # (n-1)(var1+var2)
    pstddnm = nx + ny - 2                                   # (n-1)*2
    pstd = (pstdnmr / pstddnm)**(0.5)
    cohen = (x.mean() - y.mean()) / pstd
    glass = (x.mean() - y.mean()) / x.std(ddof=1)
    hedges = cohen * (1 - 3/(4 * (nx+ny) - 9) )             # cohen * (1-3/(8*n-9))
    return (cohen, glass, hedges)

In [3]:
o = pd.read_excel('OptoJumpSorted.xlsx', sheet_name='GeneralSbs')
ost = pd.read_excel('OptoJumpSorted.xlsx', sheet_name='GeneralSt')

In [4]:
descr = o.loc[:,'WPX_1J1':].agg(['count', 'sum', 'median', 'mean', 'min', 'max', 'std', 'var']).T
descr.loc[:,'sum'] = 39

In [5]:
descr['kurtosis'] = o.loc[:,'WPX_1J1':].apply(st.kurtosis)
descr['skew'] = o.loc[:,'WPX_1J1':].apply(st.skew)
descr[['shapiro statistic', 'shapiro p value']] = o.loc[:,'WPX_1J1':].apply(st.shapiro).T

In [6]:
descrt = descr.T
replacements = {"1": "_final", "2": "_initial"}
descrt = descrt.rename(columns=lambda col: col[:-1] + replacements[col[-1]] if col[-1] in replacements else col)
descrt.columns = descrt.columns.str.replace('_',' ')
descr = descrt.T

In [7]:
st.t.ppf(0.025,39)

np.float64(-2.022690920036761)

In [8]:
ost.loc[:40, 'Test'] = 'final'
ost.loc[40:, 'Test'] = 'initial'

In [9]:
ost.columns = ['Index', 'Name', 'Test', 'Group', 'Sex', 'Weight', 'Height', 'WPX 1 Jump',
       'Step Width 1 Jump 1', 'Step Width 1 Jump 2', 'Flight Time 1 Jump', 'Elevation 1 Jump', 'Jumps',
       'Flight Time', 'Contact Time', 'Elevation', 'Propulsive Power', 'Pace', 'RSI', 'WPX',
       'WPY', 'Step Width', 'Distance', 'Total Time', 'Jump Time', 'Slope - Flight Time',
       'Slope - Contact Time', 'Slope - Elevation', 'Slope - Power', 'Slope - Pace', 'Slope - RSI',
       'Slope - Step Width', 'Slope - Distance', 'Flight Time BFI', 'Contact Time BFI',
       'Elevation BFI', 'Power BFI', 'Pace BFI', 'RSI BFI', 'Step Width BFI', 'Distance BFI']

In [10]:
tt = pd.DataFrame(data=0.1, index=ost.loc[:,'WPX 1 Jump':].columns,
                  columns=['Significance Level', 'df',
                           'T critical (two tails)', 'T statistic','T test p-value',
                           'Levene statistic', 'Levene p-value', 'Effect size (Cohen d)'])

In [11]:
tt.loc[:,'Significance Level'] = 0.05
tt.loc[:,'df'] = 39
tt.loc[:,'T critical (two tails)'] = 2.02

In [12]:
for col in ost.loc[:,'WPX 1 Jump':].columns:
    tt.loc[col,'T statistic'] = st.ttest_rel(ost.loc[40:,col],ost.loc[:39,col])[0]
    tt.loc[col,'T test p-value'] = st.ttest_rel(ost.loc[40:,col],ost.loc[:39,col])[1]
    tt.loc[col,'Levene statistic'] = st.levene(ost.loc[40:,col],ost.loc[:39,col])[0]
    tt.loc[col,'Levene p-value'] = st.levene(ost.loc[40:,col],ost.loc[:39,col])[1]
    tt.loc[col,'Effect size (Cohen d)'] = cohen_glass_hedges(ost.loc[40:,col],ost.loc[:39,col])[0]


In [13]:
tt

,Significance Level,df,T critical (two tails),T statistic,T test p-value,Levene statistic,Levene p-value,Effect size (Cohen d)
WPX 1 Jump,0.05,39.0,2.02,-1.142615,0.260165,4.275105,0.041991,-0.266166
Step Width 1 Jump 1,0.05,39.0,2.02,2.698646,0.010237,1.339941,0.250577,0.586261
Step Width 1 Jump 2,0.05,39.0,2.02,0.073764,0.941575,0.303828,0.583068,0.015912
Flight Time 1 Jump,0.05,39.0,2.02,1.210834,0.233246,0.522070,0.472121,0.222565
Elevation 1 Jump,0.05,39.0,2.02,1.126177,0.266973,0.181638,0.671143,0.215471
Jumps,0.05,39.0,2.02,1.272300,0.210800,0.028859,0.865545,0.298427
Flight Time,0.05,39.0,2.02,-2.692173,0.010405,0.006044,0.938234,-0.561134
Contact Time,0.05,39.0,2.02,2.064285,0.045687,1.234990,0.269853,0.335841
Elevation,0.05,39.0,2.02,-2.460740,0.018397,0.538395,0.465298,-0.552162
Propulsive Power,0.05,39.0,2.02,-2.527960,0.015634,0.131834,0.717520,-0.541697


In [14]:
descr

,count,sum,median,mean,min,max,std,var,kurtosis,skew,shapiro statistic,shapiro p value
WPX 1J final,40.0,39.0,1.050000,-0.532500,-30.800000,22.4,10.366746,107.469429,0.940999,-0.641763,0.964200,2.324930e-01
WPX 1J initial,40.0,39.0,-2.350000,-2.877500,-17.200000,16.6,6.911770,47.772558,0.584008,0.160907,0.968966,3.335955e-01
Step Width 1J 1 final,40.0,39.0,25.000000,24.530000,0.000000,32.3,5.164037,26.667282,12.119496,-3.169955,0.636733,1.029151e-08
Step Width 1J 1 initial,40.0,39.0,27.100000,26.877500,20.800000,32.3,2.323733,5.399737,0.122702,-0.167413,0.970938,3.851906e-01
Step Width 1J 2 final,40.0,39.0,28.650000,31.672500,22.900000,64.6,9.190351,84.462558,3.813755,2.010682,0.749188,6.913966e-07
...,...,...,...,...,...,...,...,...,...,...,...,...
Bosco RSI initial,40.0,39.0,90.147124,89.070321,65.783664,100.0,9.412012,88.585972,-0.265990,-0.725970,0.920017,7.710687e-03
Bosco Step Width final,40.0,39.0,65.281880,66.652865,31.354659,100.0,15.087419,227.630208,0.759705,0.558716,0.925872,1.183799e-02
Bosco Step Width initial,40.0,39.0,63.631707,68.043886,27.467105,100.0,20.380370,415.359482,-0.517814,0.129544,0.924649,1.081530e-02
Bosco Distance final,40.0,39.0,97.428291,97.445644,91.650413,100.0,1.804037,3.254550,1.399940,-0.880823,0.929697,1.574373e-02


In [15]:
descrt.to_csv('descriptivet.csv')
descr.to_csv('descriptive.csv')
tt.to_csv('tt.csv')
tt.T.to_csv('ttt.csv')